In [2]:
import numpy as np

np.linspace(0, 10, 5)

array([ 0. ,  2.5,  5. ,  7.5, 10. ])

In [3]:
%load_ext autotime

time: 105 µs (started: 2026-08-27 20:50:00 +05:30)


In [4]:
import os, json, textwrap

from dotenv import load_dotenv


# Wrap long model output at 80 columns instead of one endless line.
def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    print(textwrap.fill(text, width=80))


load_dotenv("/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found! Check the path to your openai_key.env file.")

pretty_print("API key loaded successfully.")

API key loaded successfully.
time: 4.62 ms (started: 2026-08-27 20:50:00 +05:30)


In [5]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
MODEL = "gpt-5-nano"

pretty_print("OpenAI client ready. Model:", MODEL)

OpenAI client ready. Model: gpt-5-nano
time: 673 ms (started: 2026-08-27 20:50:06 +05:30)


In [6]:
response = client.responses.create(
    model=MODEL,
    input = "What's the most up-to-date news between USA and Iran war?",
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"},
)
print(response.output_text)


I don’t have live browsing access to pull the latest news. To get the most up-to-date information on any conflict between the USA and Iran, please check reliable sources such as:

- BBC News, Reuters, Associated Press
- The New York Times, The Washington Post (regional or paywalled)
- Official statements from the U.S. Department of Defense or State Department
- Iran’s official news agency IRNA or state media

If you want, tell me a date range or what aspect you’re interested in (military actions, diplomacy, sanctions, casualties), and I can summarize typical sources or explain how to verify breaking news. If you can share a link, I can help interpret it.
time: 6.06 s (started: 2026-08-27 20:51:22 +05:30)


# Function Calling

In [7]:
def add_something(a, b):
    print(f"Adding {a} and {b} together...")
    return a + b


# These two lines are IDENTICAL as far as Python is concerned:
add_something(a=7, b=12)
add_something(**{"a": 7, "b": 12})

Adding 7 and 12 together...
Adding 7 and 12 together...


19

time: 2.92 ms (started: 2026-08-27 21:22:38 +05:30)


## 1.4 · Anatomy of a tool definition

Here's the contract you hand the model. Every field earns its place:

```python
{
    "type": "function",          # always "function" for your own tools
    "name": "add",               # what the model calls it by
    "description": "Add two numbers together.",   # helps the model decide WHEN to use it
    "parameters": {              # JSON Schema describing the arguments
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],           # which fields are mandatory
        "additionalProperties": False,    # no extra fields (required by strict mode)
    },
    "strict": True,              # enforce the schema — reject anything that doesn't match
}
```

- The **`description`** is the part the model reads to decide *whether this tool is relevant*.
  A vague description is the #1 cause of "why didn't it call my tool?"
- The **`parameters`** schema tells it *exactly what arguments to generate*.
- **`strict: True`** makes the API guarantee the arguments match your schema.

> The model never sees your Python function. It only ever sees this JSON.

Let's give it an `add` tool and watch what comes back.

In [17]:
add_tool = {
    "type": "function",
    "name": "add_schema",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

response = client.responses.create(
    model=MODEL,
    instructions="Use the add tool for any math. Never compute math yourself.",
    input="What is 7 + 12?",
    tools=[add_tool],
)


print("Output items from the model:")
print("-" * 45)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"    name      : {item.name}")
        print(f"    arguments : {item.arguments}")
        print(f"    call_id   : {item.call_id}")

print(f"\noutput_text: {response.output_text!r}   <- empty! it didn't answer, it ASKED")



Output items from the model:
---------------------------------------------
  Type: reasoning
  Type: function_call
    name      : add_schema
    arguments : {"a":7,"b":12}
    call_id   : call_qNiJwt7YPnbpLcu2fj8CYv7l

output_text: ''   <- empty! it didn't answer, it ASKED
time: 2.58 s (started: 2026-08-27 21:53:35 +05:30)


In [11]:
[item for item in response.output if item.type == "function_call"]

[ResponseFunctionToolCall(arguments='{"a":7,"b":12}', call_id='call_UNSahPMtlx5aw5gaoIU3UA3p', name='add', type='function_call', id='fc_0cff2e66b43cfee3006a905e57e74c87d2a632e3df4d1d95d2', namespace=None, status='completed')]

time: 1.07 ms (started: 2026-08-27 21:28:23 +05:30)


In [12]:
# ── Step 3: WE execute it, with the model's arguments ──
function_call = [item for item in response.output if item.type == "function_call"][0]
args = json.loads(function_call.arguments)   # '{"a":7,"b":12}'  ->  {'a': 7, 'b': 12}

args

{'a': 7, 'b': 12}

time: 811 µs (started: 2026-08-27 21:28:56 +05:30)


In [13]:
def add(a, b):
    return a + b

# ── Step 3: WE execute it, with the model's arguments ──
function_call = [item for item in response.output if item.type == "function_call"][0]
args = json.loads(function_call.arguments)   # '{"a":7,"b":12}'  ->  {'a': 7, 'b': 12}

result = add(**args)                          # the ** from section 1.3, doing real work
print(f"Step 3 — we ran add(**{args}) -> {result}")

Step 3 — we ran add(**{'a': 7, 'b': 12}) -> 19
time: 357 µs (started: 2026-08-27 21:29:14 +05:30)


In [14]:
# ── Step 4: hand the result back, quoting the call_id so it knows which call this answers ──
tool_outputs = [
    {
        "type": "function_call_output",
        "call_id": function_call.call_id,
        "output": str(result),          # must be a STRING
    }
]
print("Step 4 — sending back:", tool_outputs)

# ── Step 5: the model writes the final answer using our real number ──
# previous_response_id links this call to the earlier one, so we don't have to
# re-send the question or the model's own reasoning.
final = client.responses.create(
    model=MODEL,
    instructions="Always use the add tool for math. Never compute yourself.",
    previous_response_id=response.id,
    input=tool_outputs,
    tools=[add_tool],
)

print("Step 5 — final answer:", final.output_text)

Step 4 — sending back: [{'type': 'function_call_output', 'call_id': 'call_UNSahPMtlx5aw5gaoIU3UA3p', 'output': '19'}]
Step 5 — final answer: 19
time: 1.99 s (started: 2026-08-27 21:30:03 +05:30)


4213423 * 9238477

## Multiple Tools

In [18]:
# Four tools. Same shape as add_tool, so read one and skim the rest.
sub_tool = {
    "type": "function",
    "name": "subtract_schema",
    "description": "Subtract b from a and return the difference.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

mul_tool = {
    "type": "function",
    "name": "multiply_schema",
    "description": "Multiply two numbers together and return the product.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

div_tool = {
    "type": "function",
    "name": "divide_schema",
    "description": "Divide a by b and return the answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "Numerator (dividend)"},
            "b": {"type": "number", "description": "Denominator (divisor)"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}


def subtract(a, b):
    return a - b


def multiply(a, b):
    return a * b


def divide(a, b):
    if b == 0:
        return "Error: Division by zero"
    return a / b


# Two lookup tables we'll pass around together: what the model sees, and what we run.
TOOLS = [add_tool, sub_tool, mul_tool, div_tool]
DISPATCH = {"add_schema": add, "subtract_schema": subtract, "multiply_schema": multiply, "divide_schema": divide}

print("Offering", len(TOOLS), "tools:", list(DISPATCH))

Offering 4 tools: ['add_schema', 'subtract_schema', 'multiply_schema', 'divide_schema']
time: 801 µs (started: 2026-08-27 21:53:42 +05:30)


In [20]:
DISPATCH['add_schema'](7, 12)

19

time: 806 µs (started: 2026-08-27 21:55:02 +05:30)


In [19]:
DEV_POLICY = "Use the tools for any math. Never compute math yourself."

response = client.responses.create(
    model=MODEL,
    instructions=DEV_POLICY,
    input="What is 1247 * 83 + 19 / 3.7?",
    tools=TOOLS,
    reasoning={"effort": "minimal"},
)

print("Round 1 — the model asked for:")
for item in response.output:
    if item.type == "function_call":
        print(f"   {item.name}({item.arguments})")

Round 1 — the model asked for:
   multiply_schema({"a":1247,"b":83})
   divide_schema({"a":19,"b":3.7})
time: 2.24 s (started: 2026-08-27 21:53:42 +05:30)


In [21]:
# We run BOTH calls and send BOTH results back in one go.
tool_outputs = []
for call in response.output:
    if call.type != "function_call":
        continue
    args = json.loads(call.arguments)
    result = DISPATCH[call.name](**args)      # look the function up by name, then unpack the args
    print(f"   we ran {call.name}(**{args}) -> {result}")
    tool_outputs.append(
        {"type": "function_call_output", "call_id": call.call_id, "output": str(result)}
    )
tool_outputs

   we ran multiply_schema(**{'a': 1247, 'b': 83}) -> 103501
   we ran divide_schema(**{'a': 19, 'b': 3.7}) -> 5.135135135135135


[{'type': 'function_call_output',
  'call_id': 'call_2bXkq4Anr58azLbxY67EK2bA',
  'output': '103501'},
 {'type': 'function_call_output',
  'call_id': 'call_HgFcNYWPikldonnLAw2NAndv',
  'output': '5.135135135135135'}]

time: 1.42 ms (started: 2026-08-27 21:55:28 +05:30)


In [22]:
response = client.responses.create(
    model=MODEL,
    instructions=DEV_POLICY,
    previous_response_id=response.id,
    input=tool_outputs,
    tools=TOOLS,
    reasoning={"effort": "minimal"},
)

print("\nRound 2 — the model asked for:")
for item in response.output:
    if item.type == "function_call":
        print(f"   {item.name}({item.arguments})")


Round 2 — the model asked for:
   add_schema({"a":103501,"b":5.135135135135135})
time: 1.76 s (started: 2026-08-27 21:56:04 +05:30)


In [24]:
# One more round to finish it off.
tool_outputs = []
for call in response.output:
    if call.type != "function_call":
        continue
    args = json.loads(call.arguments)
    result = DISPATCH[call.name](**args)
    print(f"   we ran {call.name}(**{args}) -> {result}")
    tool_outputs.append(
        {"type": "function_call_output", "call_id": call.call_id, "output": str(result)}
    )

final = client.responses.create(
    model=MODEL,
    instructions=DEV_POLICY,
    previous_response_id=response.id,
    input=tool_outputs,
    tools=TOOLS,
    reasoning={"effort": "minimal"},
)

print("\nRound 3 — no more tool calls. Final answer:")
print("  ", final.output_text)
print("  Python agrees:", 1247 * 83 + 19 / 3.7)

   we ran add_schema(**{'a': 103501, 'b': 5.135135135135135}) -> 103506.13513513513

Round 3 — no more tool calls. Final answer:
   The result is 103,506.13513513513.
  Python agrees: 103506.13513513513
time: 2.63 s (started: 2026-08-27 21:58:19 +05:30)


In [26]:
tool_outputs

[{'type': 'function_call_output',
  'call_id': 'call_LHg3FFUFz2ia7gVfbmFqPpbd',
  'output': '103506.13513513513'}]

time: 827 µs (started: 2026-08-27 21:58:33 +05:30)


In [ ]:
#1247 * 83 + 19 / 3.7

In [28]:
def run_tool_loop(user_message, tools, dispatch, instructions):
    # Ask -> run whatever tools it asks for -> hand the results back -> repeat.
    # Returns the final response object:
    #   text_format given  ->  read response.output_parsed  (a validated Pydantic object)
    #   text_format None   ->  read response.output_text    (prose)
    schema = {}

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=[{"role": "user", "content": user_message}],
        tools=tools,
        **schema,
    )

    for round_no in range(1, 7):                 # hard cap, so a confused model can't spin forever
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:                            # it stopped asking -> this is the answer
            return response

        tool_outputs = []
        for call in calls:
            args = json.loads(call.arguments)
            result = dispatch[call.name](**args)
            print(f"   round {round_no}: {call.name}({args}) -> {result}")
            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(result),   # json.dumps handles dicts, numbers and strings
                }
            )

        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools
        )

    still_wants = [item.name for item in response.output if item.type == "function_call"]
    raise RuntimeError(f"Still asking for tools after 6 rounds: {still_wants}. "
                       "Usually means a tool result didn't answer the question it was asked.")

time: 1.19 ms (started: 2026-08-27 22:03:59 +05:30)


In [29]:
# The same question, now in one call.
final = run_tool_loop(
    "What is 1247 * 83 + 19 / 3.7?",
    tools=TOOLS,
    dispatch=DISPATCH,
    instructions=DEV_POLICY,
)
print("\nFinal:", final.output_text)

   round 1: multiply_schema({'a': 1247, 'b': 83}) -> 103501
   round 1: divide_schema({'a': 19, 'b': 3.7}) -> 5.135135135135135
   round 2: add_schema({'a': 103501, 'b': 5.135135135135135}) -> 103506.13513513513

Final: 103,506.135135135... (exactly 3,829,727/37)
time: 19.7 s (started: 2026-08-27 22:04:08 +05:30)


# Structured Outputs

In [ ]:
#def issue_refund(amount):
#    .......

#    issues_refun()

#issue_refun(101) $101

# Pydantic

In [30]:
from pydantic import BaseModel


# The contract: a valid Copperleaf ticket is EXACTLY these three typed fields.
class SupportTicket(BaseModel):
    category: str
    urgency: str
    refund_amount: float

model_output = {"category": "shipping", "urgency": "high", "refund_amount": 49.99}

ticket = SupportTicket.model_validate(model_output)     # the gate the model's output must pass
print("✅ validated:", ticket)

✅ validated: category='shipping' urgency='high' refund_amount=49.99
time: 997 µs (started: 2026-08-27 22:15:35 +05:30)


In [32]:
model_output = {"category": "shipping", "urgency": "high", "refund_amount": "$49.99"}

ticket = SupportTicket.model_validate(model_output)     # the gate the model's output must pass
print("✅ validated:", ticket)

ValidationError: 1 validation error for SupportTicket
refund_amount
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='$49.99', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing

time: 34.3 ms (started: 2026-08-27 22:15:54 +05:30)


In [34]:
from typing import Literal

from pydantic import Field, ValidationError


class SupportTicket(BaseModel):
    # Only these exact strings exist -> the model cannot invent a category
    category: Literal["billing", "shipping", "technical", "account", "other"]
    urgency: Literal["low", "medium", "high"]
    # Money is bounded -> the 100x decimal slip can't get through
    refund_amount: float = Field(ge=0, le=500)


# The invented category from 2.2:
try:
    SupportTicket.model_validate({"category": "damaged_item", "urgency": "high", "refund_amount": 49.99})
except ValidationError as e:
    print("🚫 bad category :", e.errors()[0]["msg"])

# The EXACT payload from the silent-bug cell that wired $5,249:
try:
    SupportTicket.model_validate({"category": "shipping", "urgency": "high", "refund_amount": 4999})
except ValidationError as e:
    print("🚫 absurd amount:", e.errors()[0]["msg"])

# A legitimate ticket still passes untouched:
print("✅", SupportTicket.model_validate(
    {"category": "shipping", "urgency": "high", "refund_amount": 49.99}))

🚫 bad category : Input should be 'billing', 'shipping', 'technical', 'account' or 'other'
🚫 absurd amount: Input should be less than or equal to 500
✅ category='shipping' urgency='high' refund_amount=49.99
time: 3.13 ms (started: 2026-08-27 22:18:17 +05:30)


In [35]:
from pydantic import field_validator


class Customer(BaseModel):
    name: str
    email: str


class SupportTicket(BaseModel):
    category: Literal["billing", "shipping", "technical", "account", "other"]
    urgency: Literal["low", "medium", "high"]
    refund_amount: float = Field(ge=0, le=500)
    order_id: str = Field(description="The 5-digit Copperleaf order number")
    customer: Customer                                   # <- a nested model

    @field_validator("order_id")
    @classmethod
    def must_be_5_digits(cls, v):
        if not (v.isdigit() and len(v) == 5):
            raise ValueError(f"order_id must be exactly 5 digits, got {v!r}")
        return v



time: 1.53 ms (started: 2026-08-27 22:20:03 +05:30)


In [37]:
# One messy, realistic customer email we'll reuse for the whole of Part 2.
# Note the signature — we'll need the name and address later.
customer_email = (
    "hey, my order 10432 turned up completely smashed — the ceramic mug is in pieces. "
    "I paid 49.99 for it and honestly I'm pretty annoyed, this is the second time. "
    "I just want my money back asap, can you sort this out today?\n\n"
    "Thanks, Sam Rivera (sam.rivera@example.com)"
)
print(customer_email)

hey, my order 10432 turned up completely smashed — the ceramic mug is in pieces. I paid 49.99 for it and honestly I'm pretty annoyed, this is the second time. I just want my money back asap, can you sort this out today?

Thanks, Sam Rivera (sam.rivera@example.com)
time: 246 µs (started: 2026-08-27 22:20:50 +05:30)


In [41]:
response = client.responses.parse(
    model=MODEL,
    instructions = " I want you to print just hello, nothing else",
    input=[
        {"role": "system", "content":
            "Extract the support ticket from the customer message. Infer urgency from the tone. "
            "refund_amount is what the customer paid / wants back (0 if they aren't asking for one)."},
        {"role": "user", "content": customer_email},
    ],
    text_format=SupportTicket,          # <- hand the model OUR class
)

ticket = response.output_parsed          # <- already a validated SupportTicket, not text
print("Type of result:", type(ticket).__name__)
print(ticket, "\n")
print("category :", ticket.category, " (a guaranteed-valid Literal)")
print("urgency  :", ticket.urgency)
print("refund   :", ticket.refund_amount, "->", type(ticket.refund_amount).__name__)
print("order id :", ticket.order_id)
print("customer :", ticket.customer.name, "<" + ticket.customer.email + ">")

Type of result: SupportTicket
category='shipping' urgency='high' refund_amount=49.99 order_id='10432' customer=Customer(name='Sam Rivera', email='sam.rivera@example.com') 

category : shipping  (a guaranteed-valid Literal)
urgency  : high
refund   : 49.99 -> float
order id : 10432
customer : Sam Rivera <sam.rivera@example.com>
time: 8.65 s (started: 2026-08-27 22:22:47 +05:30)


https://open-meteo.com/

https://www.geoapify.com/geocoding-api/

The homework is that you have to design a function that can basically, given a coordinate or given a name, can give you the temperature and weather for that location. So the language model should be able to use this function to design a tool call accordingly.